# L-0 Step 0：LLM 直接標注種子庫建立

不生成合成日報。直接從真實日報抽 5,000 篇送 Gemini Flash 標注 L 層 + 信心分數，
高信心結果向量化後作為 KNN 種子庫。

| 子任務 | 規格 |
|--------|------|
| A 日報抽樣 | SQL Server CRMGY，5,000 篇，排除短文本/MA/廢止，按月份分層抽樣 |
| B Gemini Flash 標注 | `{l_layer, confidence, key_signals}`；溫度 0.1；14 RPM；斷點續跑 |
| C 高信心篩選 | confidence ≥ 0.80；每 L 層最多 350 篇平衡抽樣 |
| D 問卷語料合併 | ~541,537 筆，格式 `[L層] 答案文字`，與日報種子合併 |
| E 向量化 | text-embedding-004；格式 `[L3][Day120] 摘要`；Vertex AI Vector Search |
| 完成標準 | 每 L 層 ≥ 200 高信心種子；KNN Top-1 ≥ 75%；uncertain < 20% |

## 0. 安裝套件

In [151]:
!pip install google-genai google-cloud-aiplatform pyodbc pandas tqdm -q

## 1. 匯入套件 & 全域設定

In [152]:
import os, re, json, csv, time, hashlib, configparser
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pyodbc
import pandas as pd
from google import genai
from tqdm import tqdm

In [153]:
# ── SQL Server ───────────────────────────────
SQL_INI_PATH = r"D:\yujui\SqlServer18.txt"
SQL_DATABASE = "DSC_CRM_SP"

# ── Google Cloud ─────────────────────────────
GCP_INI_PATH    = r"D:\yujui\GoogleCloud.ini"
GEMINI_MODEL    = "gemini-2.0-flash"
EMBED_MODEL     = "gemini-embedding-2-preview"
VERTEX_LOCATION = "asia-east1"

# ── 抽樣 ─────────────────────────────────────
TARGET_N         = 5000
MIN_LOG_LEN      = 30
EXCLUDE_KEYWORDS = ["MA簽回", "MA簽約", "維護合約", "廢止", "解散"]

# ── 標注 ─────────────────────────────────────
TEMPERATURE       = 0.1
REQUESTS_PER_MIN  = 14       # 保守，低於免費層 15 RPM
MAX_WORKERS       = 8
CONFIDENCE_THRESH = 0.80
MAX_PER_LAYER     = 350      # 每 L 層種子上限，防高頻層壓低頻層

# ── 問卷語料 ──────────────────────────────────
SURVEY_CSV = r"D:\yujui\痛點需求地圖\survey_corpus.csv"  # 欄位：l_layer, answer_text

# ── 輸出 ─────────────────────────────────────
OUTPUT_DIR          = Path(r"D:\yujui\痛點需求地圖\seed_output")
LABELED_JSONL       = OUTPUT_DIR / "labeled_logs.jsonl"
SEED_VECTORS_JSONL  = OUTPUT_DIR / "seed_vectors.jsonl"
PROGRESS_FILE       = OUTPUT_DIR / "progress.json"
QUALITY_REPORT_CSV  = OUTPUT_DIR / "quality_report.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"輸出目錄：{OUTPUT_DIR}")

輸出目錄：D:\yujui\痛點需求地圖\seed_output


## 2. 建立連線

In [154]:
# ── SQL Server ───────────────────────────────
cfg_sql = configparser.ConfigParser()
cfg_sql.read(SQL_INI_PATH)
if 'mssql' not in cfg_sql:
    raise RuntimeError(f"找不到 [mssql] 區段：{SQL_INI_PATH}")
sec = cfg_sql['mssql']
conn = pyodbc.connect(
    f"DRIVER={{SQL Server}};SERVER={sec['server']};DATABASE={SQL_DATABASE};"
    f"UID={sec['uid']};PWD={sec['pwd']};", autocommit=True
)
print(f"SQL Server 連線成功：{sec['server']}")

# ── Gemini ───────────────────────────────────
cfg_gc = configparser.ConfigParser()
cfg_gc.read(GCP_INI_PATH)
svc_key = os.getenv("GOOGLE_APPLICATION_CREDENTIALS") or cfg_gc.get("gcp", "service_account_key", fallback="")
if svc_key:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = svc_key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or cfg_gc.get("gcp", "gemini_api_key", fallback="")
PROJECT_ID     = os.getenv("PROJECT_ID")     or cfg_gc.get("gcp", "project_id",      fallback="digiwin-ai")
client = genai.Client(api_key=GEMINI_API_KEY)
print(f"Gemini 模型：{GEMINI_MODEL}")
print(f"GCP Project：{PROJECT_ID}")

SQL Server 連線成功：10.20.99.18
Gemini 模型：gemini-2.0-flash
GCP Project：digiwin-ai


---
## 子任務 A：日報抽樣
從 CRMGY 按月份分層抽取 5,000 篇，排除短文本與無效日報。

In [155]:
exclude_sql = " AND ".join([f"GY012 NOT LIKE '%{kw}%'" for kw in EXCLUDE_KEYWORDS])

# 查各月份可用筆數
dist_df = pd.read_sql(f"""
    SELECT FORMAT(CREATE_DATE,'yyyy-MM') AS ym, COUNT(*) AS total
    FROM CRMGY
    WHERE GY012 IS NOT NULL
      AND LEN(LTRIM(RTRIM(GY012))) >= {MIN_LOG_LEN}
      AND {exclude_sql}
    GROUP BY FORMAT(CREATE_DATE,'yyyy-MM')
    ORDER BY ym
""", conn)

# 按比例計算每月配額
total_avail = dist_df["total"].sum()
dist_df["quota"] = (dist_df["total"] / total_avail * TARGET_N).round().astype(int).clip(lower=1)
diff = TARGET_N - dist_df["quota"].sum()
dist_df.loc[dist_df["total"].idxmax(), "quota"] += diff

print(f"有效月份：{len(dist_df)} 個，總可用：{total_avail:,} 筆，配額合計：{dist_df['quota'].sum()}")
dist_df.tail(10)

C:\Users\DSC\AppData\Local\Temp\ipykernel_33596\2525813425.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dist_df = pd.read_sql(f"""


有效月份：303 個，總可用：1,960,819 筆，配額合計：5000


,ym,total,quota
293,2025-07,10544,27
294,2025-08,9201,23
295,2025-09,9888,25
296,2025-10,10029,26
297,2025-11,9360,24
298,2025-12,9412,24
299,2026-01,9751,25
300,2026-02,4070,10
301,2026-03,12182,31
302,2026-04,2893,7


In [156]:
# 逐月隨機抽取
all_samples = []
for _, row in tqdm(dist_df.iterrows(), total=len(dist_df), desc="逐月抽樣"):
    month_df = pd.read_sql(f"""
        SELECT TOP {int(row['quota'])}
            GY001 AS company_id, GY003 AS salesperson_id,
            GY004 AS contact_date, GY022 AS log_type,
            GY012 AS log_content, CREATE_DATE,
            LEN(GY012) AS log_len,
            FORMAT(CREATE_DATE,'yyyy-MM') AS ym
        FROM CRMGY
        WHERE GY012 IS NOT NULL
          AND LEN(LTRIM(RTRIM(GY012))) >= {MIN_LOG_LEN}
          AND {exclude_sql}
          AND FORMAT(CREATE_DATE,'yyyy-MM') = '{row['ym']}'
        ORDER BY NEWID()
    """, conn)
    all_samples.append(month_df)

sample_df = pd.concat(all_samples, ignore_index=True)
sample_df["log_content"] = sample_df["log_content"].astype(str).str.replace("\n", " ", regex=False)
sample_df = sample_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n抽樣完成：{len(sample_df):,} 筆")
print(f"字數範圍：{sample_df['log_len'].min()} ~ {sample_df['log_len'].max()} 字（平均 {sample_df['log_len'].mean():.0f}）")

逐月抽樣:   0%|          | 0/303 [00:00<?, ?it/s]C:\Users\DSC\AppData\Local\Temp\ipykernel_33596\2780868751.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  month_df = pd.read_sql(f"""
逐月抽樣:   0%|          | 1/303 [00:16<1:22:32, 16.40s/it]C:\Users\DSC\AppData\Local\Temp\ipykernel_33596\2780868751.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  month_df = pd.read_sql(f"""
逐月抽樣: 100%|██████████| 303/303 [54:04<00:00, 10.71s/it] 


抽樣完成：4,999 筆
字數範圍：30 ~ 1020 字（平均 75）



C:\Users\DSC\AppData\Local\Temp\ipykernel_33596\2780868751.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sample_df = pd.concat(all_samples, ignore_index=True)


---
## 子任務 B：Gemini Flash 標注
溫度 0.1；14 RPM 速率控制；支援斷點續跑；費用約 $0.15–0.35。

In [157]:
SYSTEM_PROMPT = """
你是一位 B2B 銷售數據分析專家。判斷業務日報屬於哪個銷售週期階段（L 層）並給出信心分數。

L1 起因/阻礙｜最初期｜阻礙、卡關、問題、無法推進、第一次、困難
L2 角色壓力｜探索期｜老闆、主管、壓力、KPI、部門、決策鏈、績效考核
L3 戰略目標｜需求確認期｜轉型、目標、策略、計畫、三年、未來、佈局
L4 產業議題｜深化期｜產業、趨勢、法規、政策、競爭、市場、ESG、供應鏈
L5 問項精煉｜方案探索期｜評估、比較、需求確認、想了解、能不能、標準是什麼
L6 動態屬性｜決策臨界期｜態度、溫度、變化、猶豫、轉變、比上次、感覺
L7 實戰對策｜成交結案期｜合約、簽訂、成交、付款、方案確認、價格談、折扣

輸出規則：只輸出 JSON，不加任何說明。混雜多層時選時間序列最晚的。無法判斷填 uncertain。

{"l_layer":"L1","confidence":0.92,"reasoning":"30字以內","key_signals":["卡關","第一次"]}
"""

def clean_json(text: str) -> str:
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return m.group(0).replace("True","true").replace("False","false") if m else "{}"

def label_l_layer(log_text: str) -> dict:
    prompt = f"{SYSTEM_PROMPT}\n日報：{str(log_text)[:2000]}"
    try:
        resp = client.models.generate_content(
            model=GEMINI_MODEL, contents=prompt,
            config={"temperature": TEMPERATURE, "max_output_tokens": 200}
        )
        parsed = json.loads(clean_json(resp.text))
        return {
            "l_layer":     parsed.get("l_layer", "uncertain"),
            "confidence":  float(parsed.get("confidence", 0.0)),
            "reasoning":   parsed.get("reasoning", ""),
            "key_signals": parsed.get("key_signals", []),
        }
    except Exception as e:
        return {"l_layer": "uncertain", "confidence": 0.0, "reasoning": str(e)[:50], "key_signals": []}

In [158]:
# ── 斷點續跑 ──────────────────────────────────
def load_progress():
    if PROGRESS_FILE.exists():
        return set(json.loads(PROGRESS_FILE.read_text())["done"])
    return set()

def save_progress(done: set):
    PROGRESS_FILE.write_text(json.dumps({"done": list(done), "updated_at": datetime.now().isoformat()}))

# ── 單筆測試 ──────────────────────────────────
test = label_l_layer(sample_df.iloc[0]["log_content"])
print("單筆測試結果：")
print(json.dumps(test, ensure_ascii=False, indent=2))

單筆測試結果：
{
  "l_layer": "L1",
  "confidence": 0.85,
  "reasoning": "客戶對簽核安全性極為重視，需要提供相關資料，顯示專案推進遇到阻礙，處於初期階段。",
  "key_signals": [
    "阻礙",
    "卡關"
  ]
}


In [159]:
# ── 批次標注主流程（14 RPM 速率控制）─────────────
RESUME = False  # True = 從上次中斷處繼續

done_ids = load_progress() if RESUME else set()
todo = sample_df[~sample_df.index.astype(str).isin(done_ids)].copy()
print(f"待標注：{len(todo)} 筆（已完成：{len(done_ids)} 筆）")

stats = {"total": 0, "uncertain": 0, "error": 0}
rpm_start, rpm_count = time.time(), 0

with open(LABELED_JSONL, "a", encoding="utf-8") as f:
    for idx, row in tqdm(todo.iterrows(), total=len(todo), desc="標注進度"):
        # 14 RPM 控制
        rpm_count += 1
        if rpm_count >= REQUESTS_PER_MIN:
            elapsed = time.time() - rpm_start
            if elapsed < 60:
                time.sleep(61 - elapsed)
            rpm_start, rpm_count = time.time(), 0

        label = label_l_layer(row["log_content"])
        stats["total"] += 1
        if label["l_layer"] == "uncertain": stats["uncertain"] += 1
        if label["confidence"] == 0.0:      stats["error"] += 1

        record = {
            "event_id":      hashlib.sha256(row["log_content"].encode()).hexdigest()[:16],
            "company_id":    row["company_id"],
            "ym":            row["ym"],
            "log_len":       int(row["log_len"]),
            "log_content":   row["log_content"],
            "l_layer":       label["l_layer"],
            "confidence":    label["confidence"],
            "reasoning":     label["reasoning"],
            "key_signals":   label["key_signals"],
            "labeled_at":    datetime.now().isoformat(),
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

        done_ids.add(str(idx))
        if stats["total"] % 100 == 0:
            save_progress(done_ids)

save_progress(done_ids)
uncertain_pct = stats['uncertain'] / stats['total'] * 100 if stats['total'] else 0
print(f"\n標注完成：{stats['total']} 筆｜uncertain：{stats['uncertain']} ({uncertain_pct:.1f}%)｜error：{stats['error']}")

待標注：4999 筆（已完成：0 筆）


標注進度: 100%|██████████| 4999/4999 [6:03:02<00:00,  4.36s/it]   


標注完成：4999 筆｜uncertain：12 (0.2%)｜error：7


---
## 子任務 C：高信心篩選（自適應門檻）
主門檻 confidence ≥ 0.80；若某層不足 200 篇，自動逐步降低門檻（0.75 → 0.70 → 0.65 → 0.60）直到補足或耗盡。
每 L 層最多 350 篇，防止高頻層壓低頻層。

In [160]:
# 載入標注結果
labeled = [json.loads(l) for l in open(LABELED_JSONL, encoding="utf-8") if l.strip()]
labeled_df = pd.DataFrame(labeled)

# ── 自適應門檻：不足層逐步降低 ───────────────────
FALLBACK_THRESHOLDS = [0.80, 0.75, 0.70, 0.65, 0.60]  # 依序嘗試
MIN_PER_LAYER       = 200

L_LAYERS = ["L1","L2","L3","L4","L5","L6","L7"]
seed_parts = []
threshold_log = {}  # 記錄每層實際使用的門檻

for layer in L_LAYERS:
    layer_df = labeled_df[labeled_df["l_layer"] == layer].copy()
    chosen_thresh = None
    selected = pd.DataFrame()

    for thresh in FALLBACK_THRESHOLDS:
        candidates = layer_df[layer_df["confidence"] >= thresh]
        if len(candidates) >= MIN_PER_LAYER or thresh == FALLBACK_THRESHOLDS[-1]:
            # 達標或已到最低門檻，取最多 MAX_PER_LAYER 筆（優先高信心）
            selected = (
                candidates
                .sort_values("confidence", ascending=False)
                .head(MAX_PER_LAYER)
            )
            chosen_thresh = thresh
            break

    threshold_log[layer] = {"thresh": chosen_thresh, "count": len(selected)}
    seed_parts.append(selected)

seed_df = pd.concat(seed_parts, ignore_index=True)

# ── 結果報告 ──────────────────────────────────
print(f"平衡後種子庫：{len(seed_df):,} 筆\n")
print(f"{'L層':<6} {'門檻':>6} {'篇數':>6}  狀態")
print("─" * 32)
all_pass = True
for layer in L_LAYERS:
    info   = threshold_log[layer]
    cnt    = info["count"]
    thresh = info["thresh"]
    flag   = "✅" if cnt >= MIN_PER_LAYER else "⚠️ 不足200（已取全部）"
    note   = f"（降至 {thresh}）" if thresh < CONFIDENCE_THRESH else ""
    print(f"  {layer:<4} {thresh:>6.2f} {cnt:>6}  {flag}{note}")
    if cnt < MIN_PER_LAYER:
        all_pass = False

print()
print(f"整體判斷：{'✅ 全層達標' if all_pass else '⚠️ 部分層不足，建議擴大抽樣（見下方建議）'}")

if not all_pass:
    short = [l for l in L_LAYERS if threshold_log[l]["count"] < MIN_PER_LAYER]
    total_in_labeled = {l: int((labeled_df["l_layer"]==l).sum()) for l in short}
    print(f"\n不足層在標注結果中的原始筆數：")
    for l in short:
        n = total_in_labeled[l]
        need = MIN_PER_LAYER - threshold_log[l]["count"]
        print(f"  {l}：標注共 {n} 筆，仍缺 {need} 篇 → 建議再抽 {need * 10} 筆日報重新標注")

平衡後種子庫：2,195 筆

L層         門檻     篇數  狀態
────────────────────────────────
  L1     0.80    350  ✅
  L2     0.80    282  ✅
  L3     0.75    350  ✅（降至 0.75）
  L4     0.80    238  ✅
  L5     0.80    350  ✅
  L6     0.80    275  ✅
  L7     0.80    350  ✅

整體判斷：✅ 全層達標


---
## 子任務 D：問卷語料合併
來源：SQL Server `OESTE`（問卷作答）+ `OESTA`（問項定義）+ `OESTB`（問卷活動）  
流程：撈取 → LLM 標注 L 層 → 篩選高信心 → 格式化 `[L層] 答案文字` → 合併日報種子

---
## 子任務 C-補：不足層關鍵字補抽 & 再標注
L3/L4/L6 在隨機抽樣中出現率低，改用各層關鍵字過濾，提高命中率後再補標注。

In [161]:
# ── C-補1：設定各不足層的關鍵字與補抽量（第二輪）──────
SUPPLEMENT_LAYERS = {
    "L2": {
        "need":     100,
        "keywords": ["老闆", "主管", "KPI", "績效", "部門", "決策", "長官", "董事長",
                     "壓力", "考核", "目標達成", "業績壓力", "上級", "主席"],
        "fetch_n":  800,
    },
    "L3": {
        "need":     38,
        "keywords": ["轉型", "策略", "三年計畫", "未來", "佈局", "發展方向", "長期目標",
                     "年度目標", "願景", "路線圖", "中長期", "企業目標"],
        "fetch_n":  350,
    },
    "L4": {
        "need":     58,
        "keywords": ["產業", "法規", "政策", "競爭", "市場", "ESG", "供應鏈", "外部環境",
                     "產業鏈", "同業", "景氣", "總體經濟", "碳排", "國際情勢"],
        "fetch_n":  550,
    },
    "L6": {
        "need":     81,
        "keywords": ["態度", "溫度", "猶豫", "轉變", "比上次", "突然", "感覺", "進展", "變化",
                     "積極", "冷淡", "主動", "回應速度", "明顯", "不一樣", "轉機", "有意願"],
        "fetch_n":  1800,
    },
}

labeled_ids = set(
    json.loads(l)["event_id"]
    for l in open(LABELED_JSONL, encoding="utf-8") if l.strip()
)
print(f"已標注 event_id 數：{len(labeled_ids):,}（補抽時會排除）")
for layer, cfg_s in SUPPLEMENT_LAYERS.items():
    print(f"  {layer}：缺 {cfg_s['need']} 篇，計劃補抽 {cfg_s['fetch_n']} 筆")

已標注 event_id 數：23,608（補抽時會排除）
  L2：缺 100 篇，計劃補抽 800 筆
  L3：缺 38 篇，計劃補抽 350 筆
  L4：缺 58 篇，計劃補抽 550 筆
  L6：缺 81 篇，計劃補抽 1800 筆


In [162]:
# ── C-補2：各層關鍵字補抽 ────────────────────────────
supp_frames = []

for layer, cfg_s in SUPPLEMENT_LAYERS.items():
    # 組合 OR 條件：任一關鍵字命中即可
    kw_or = " OR ".join([f"GY012 LIKE '%{kw}%'" for kw in cfg_s["keywords"]])
    # 排除關鍵字（同主流程）
    excl  = " AND ".join([f"GY012 NOT LIKE '%{kw}%'" for kw in EXCLUDE_KEYWORDS])

    q = f"""
    SELECT TOP {cfg_s['fetch_n']}
        GY001 AS company_id, GY003 AS salesperson_id,
        GY004 AS contact_date, GY022 AS log_type,
        GY012 AS log_content, CREATE_DATE,
        LEN(GY012) AS log_len,
        FORMAT(CREATE_DATE,'yyyy-MM') AS ym
    FROM CRMGY
    WHERE GY012 IS NOT NULL
      AND LEN(LTRIM(RTRIM(GY012))) >= {MIN_LOG_LEN}
      AND {excl}
      AND ({kw_or})
    ORDER BY NEWID()
    """
    df_layer = pd.read_sql(q, conn)
    df_layer["log_content"] = df_layer["log_content"].astype(str).str.replace("\n", " ", regex=False)

    # 過濾已標注
    df_layer["event_id"] = df_layer["log_content"].apply(
        lambda t: hashlib.sha256(t.encode()).hexdigest()[:16]
    )
    df_layer = df_layer[~df_layer["event_id"].isin(labeled_ids)]
    supp_frames.append(df_layer)

    print(f"  {layer}：補抽 {len(df_layer)} 筆（關鍵字：{cfg_s['keywords'][:3]}…）")

supp_df = pd.concat(supp_frames, ignore_index=True)
print(f"\n補抽合計：{len(supp_df):,} 筆")

C:\Users\DSC\AppData\Local\Temp\ipykernel_33596\3701266115.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_layer = pd.read_sql(q, conn)


  L2：補抽 786 筆（關鍵字：['老闆', '主管', 'KPI']…）


C:\Users\DSC\AppData\Local\Temp\ipykernel_33596\3701266115.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_layer = pd.read_sql(q, conn)


  L3：補抽 322 筆（關鍵字：['轉型', '策略', '三年計畫']…）


C:\Users\DSC\AppData\Local\Temp\ipykernel_33596\3701266115.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_layer = pd.read_sql(q, conn)


  L4：補抽 506 筆（關鍵字：['產業', '法規', '政策']…）


C:\Users\DSC\AppData\Local\Temp\ipykernel_33596\3701266115.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_layer = pd.read_sql(q, conn)


  L6：補抽 1625 筆（關鍵字：['態度', '溫度', '猶豫']…）

補抽合計：3,239 筆


In [163]:
# ── C-補3：補抽日報送 LLM 標注，結果 append 進 labeled_logs.jsonl ──
rpm_start, rpm_count = time.time(), 0

with open(LABELED_JSONL, "a", encoding="utf-8") as f:
    for idx, row in tqdm(supp_df.iterrows(), total=len(supp_df), desc="補抽標注"):
        rpm_count += 1
        if rpm_count >= REQUESTS_PER_MIN:
            elapsed = time.time() - rpm_start
            if elapsed < 60:
                time.sleep(61 - elapsed)
            rpm_start, rpm_count = time.time(), 0

        label = label_l_layer(row["log_content"])
        record = {
            "event_id":    row["event_id"],
            "company_id":  row["company_id"],
            "ym":          row["ym"],
            "log_len":     int(row["log_len"]),
            "log_content": row["log_content"],
            "l_layer":     label["l_layer"],
            "confidence":  label["confidence"],
            "reasoning":   label["reasoning"],
            "key_signals": label["key_signals"],
            "labeled_at":  datetime.now().isoformat(),
            "source":      "supplement",   # 標記為補抽
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        labeled_ids.add(row["event_id"])

print(f"補抽標注完成，累計已標注：{len(labeled_ids):,} 筆")

補抽標注: 100%|██████████| 3239/3239 [3:55:09<00:00,  4.36s/it]  

補抽標注完成，累計已標注：26,843 筆


In [164]:
# ── C-補4：重新跑子任務 C 篩選，確認各層是否已補足 ───
labeled_all = [json.loads(l) for l in open(LABELED_JSONL, encoding="utf-8") if l.strip()]
labeled_df  = pd.DataFrame(labeled_all)

# 補抽筆數統計（source 欄位存在才算）
has_source = "source" in labeled_df.columns

seed_parts, threshold_log = [], {}
for layer in L_LAYERS:
    layer_df = labeled_df[labeled_df["l_layer"] == layer].copy()
    for thresh in FALLBACK_THRESHOLDS:
        candidates = layer_df[layer_df["confidence"] >= thresh]
        if len(candidates) >= MIN_PER_LAYER or thresh == FALLBACK_THRESHOLDS[-1]:
            selected = candidates.sort_values("confidence", ascending=False).head(MAX_PER_LAYER)
            supp_cnt = int((selected["source"] == "supplement").sum()) if has_source else 0
            threshold_log[layer] = {"thresh": thresh, "count": len(selected), "supp": supp_cnt}
            seed_parts.append(selected)
            break

seed_df = pd.concat(seed_parts, ignore_index=True)

print(f"補抽後種子庫：{len(seed_df):,} 筆\n")
print(f"{'L層':<6} {'門檻':>6} {'篇數':>6} {'補抽':>6}  狀態")
print("─" * 42)
all_pass = True
for layer in L_LAYERS:
    info   = threshold_log[layer]
    cnt, thresh, supp = info["count"], info["thresh"], info["supp"]
    flag   = "✅" if cnt >= MIN_PER_LAYER else f"❌ 仍缺 {MIN_PER_LAYER - cnt} 篇"
    note   = f"（降至 {thresh}）" if thresh < CONFIDENCE_THRESH else ""
    print(f"  {layer:<4} {thresh:>6.2f} {cnt:>6} {supp:>6}  {flag}{note}")
    if cnt < MIN_PER_LAYER:
        all_pass = False

print()
print(f"整體判斷：{'✅ 全層達標' if all_pass else '⚠️ 仍有不足層，請調高 fetch_n 後重跑 C-補1~3'}")

補抽後種子庫：2,375 筆

L層         門檻     篇數     補抽  狀態
──────────────────────────────────────────
  L1     0.80    350    169  ✅
  L2     0.80    348    202  ✅
  L3     0.75    350    223  ✅（降至 0.75）
  L4     0.80    291    211  ✅
  L5     0.80    350     71  ✅
  L6     0.80    336    227  ✅
  L7     0.80    350     47  ✅

整體判斷：✅ 全層達標


In [165]:
# ── D1：從 SQL Server 撈取問卷資料 ──────────────
survey_query = """
SELECT DISTINCT
     [E].[客戶姓名]
    ,[E].[潛客代號]
    ,[公司名稱]   = [IG].IG018
    ,[Email]     = ISNULL([IG].IG007, [IG2].IG007)
    ,[客戶電話]   = [IG].IG010
    ,[客戶手機]   = [IG].IG011
    ,[客戶職稱]   = (SELECT GA003 FROM [DSC_CRM_SP].[dbo].CRMGA WHERE GA001='d2' AND GA002=[IG].[IG009])
    ,[客戶職能]   = (SELECT GA003 FROM [DSC_CRM_SP].[dbo].CRMGA WHERE GA001='d3' AND GA002=[IG].[IG029])
    ,[E].[活動日期]
    ,IA.IA003    AS [活動名稱]
    ,[A].[問項]
    ,[A].[問項類型]
    ,[答案] = (CASE [E].[答案(拆開)]
        WHEN '1'  THEN [A].選項1  WHEN '2'  THEN [A].選項2
        WHEN '3'  THEN [A].選項3  WHEN '4'  THEN [A].選項4
        WHEN '5'  THEN [A].選項5  WHEN '6'  THEN [A].選項6
        WHEN '7'  THEN [A].選項7  WHEN '8'  THEN [A].選項8
        WHEN '9'  THEN [A].選項9  WHEN '10' THEN [A].選項10
        WHEN '11' THEN [A].選項11 WHEN '12' THEN [A].選項12
        ELSE [E].[答案(拆開)] END)
FROM (
    SELECT [活動代號]=OESTB013,[問卷代號]=OESTB001,
           [問卷生效日]=OESTB010,[問卷使用場次]=OESTB014
    FROM Quiz..OESTB
) [B]
LEFT JOIN (
    SELECT [問卷代號]=OESTA019,[問項代號]=OESTA001,[問項排序]=OESTA027,
        [問項類型]=(CASE OESTA005 WHEN '2' THEN '單選' WHEN '3' THEN '多選' WHEN '4' THEN '問答' END),
        [問項]=OESTA009,
        OESTA012 [選項1], OESTA013 [選項2], OESTA014 [選項3], OESTA015 [選項4],
        OESTA016 [選項5], OESTA017 [選項6], OESTA020 [選項7], OESTA021 [選項8],
        OESTA022 [選項9], OESTA023 [選項10],OESTA024 [選項11],OESTA025 [選項12]
    FROM Quiz..OESTA
) [A] ON [A].[問卷代號] = [B].[問卷代號]
LEFT JOIN (
    SELECT [E].*,
           [答案(拆開)] = LTRIM(RTRIM([答案(拆開)].value))
    FROM (
        SELECT [問卷代號]=OESTE001,[問項代號]=OESTE003,
            [客戶姓名]=OESTE041,[潛客代號]=OESTE040,
            [活動日期]=OESTE042,[場次]=OESTE043,
            [問項類型]=(CASE OESTE004 WHEN '2' THEN '單選' WHEN '3' THEN '多選' WHEN '4' THEN '問答' END),
            [答案]=(CASE
                WHEN OESTE004='2' AND ISNULL(OESTE017,'')!='' THEN ISNULL(OESTE017,'')
                WHEN OESTE004='3' AND ISNULL(OESTE018,'')!='' THEN ISNULL(OESTE018,'')
                WHEN OESTE004 IN ('2','3') THEN
                    (CASE WHEN ISNULL(OESTE028,'')='1' THEN '1' ELSE '' END)+','+(CASE WHEN ISNULL(OESTE029,'')='1' THEN '2' ELSE '' END)+','+(CASE WHEN ISNULL(OESTE030,'')='1' THEN '3' ELSE '' END)+','+(CASE WHEN ISNULL(OESTE031,'')='1' THEN '4' ELSE '' END)+','+(CASE WHEN ISNULL(OESTE032,'')='1' THEN '5' ELSE '' END)+','+(CASE WHEN ISNULL(OESTE033,'')='1' THEN '6' ELSE '' END)+','+(CASE WHEN ISNULL(OESTE034,'')='1' THEN '7' ELSE '' END)+','+(CASE WHEN ISNULL(OESTE035,'')='1' THEN '8' ELSE '' END)+','+(CASE WHEN ISNULL(OESTE036,'')='1' THEN '9' ELSE '' END)+','+(CASE WHEN ISNULL(OESTE037,'')='1' THEN '10' ELSE '' END)+','+(CASE WHEN ISNULL(OESTE038,'')='1' THEN '11' ELSE '' END)+','+(CASE WHEN ISNULL(OESTE039,'')='1' THEN '12' ELSE '' END)
                WHEN OESTE004='4' THEN ISNULL(OESTE019,'')
                ELSE '' END)
        FROM Quiz..OESTE
    ) [E]
    CROSS APPLY STRING_SPLIT([E].答案, ',') [答案(拆開)]
    WHERE ISNULL([答案(拆開)].value,'') != ''
) [E] ON [E].[問卷代號]=[A].[問卷代號] AND [E].[問項代號]=[A].[問項代號]
JOIN DSC_CRM_SP..CRMIA [IA] ON [IA].IA001 = [B].[活動代號]
LEFT JOIN DSC_CRM_SP..CRMIG [IG]  ON [IG].IG001=[B].[活動代號]  AND [IG].IG002=[E].[活動日期]  AND [IG].IG003=[E].[場次] AND [IG].IG004=[E].[潛客代號] AND [IG].IG005=[E].[客戶姓名]
LEFT JOIN DSC_CRM_SP..CRMIG [IG2] ON [IG2].IG001=[B].[活動代號] AND [IG2].IG002=[E].[活動日期] AND [IG2].IG003=[E].[場次] AND [IG2].IG005=[E].[客戶姓名]
"""

print("正在從 SQL Server 撈取問卷資料...")
survey_raw = pd.read_sql(survey_query, conn)

# 清理：過濾空答案、換行
survey_raw = survey_raw[survey_raw["答案"].notna() & (survey_raw["答案"].str.strip() != "")]
survey_raw["答案"] = survey_raw["答案"].astype(str).str.replace("\n", " ", regex=False)

print(f"問卷資料撈取完成：{len(survey_raw):,} 筆")
print(f"問項類型分布：\n{survey_raw['問項類型'].value_counts().to_string()}")
survey_raw.head(3)

正在從 SQL Server 撈取問卷資料...


C:\Users\DSC\AppData\Local\Temp\ipykernel_33596\3680731504.py:64: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  survey_raw = pd.read_sql(survey_query, conn)


問卷資料撈取完成：1,170,040 筆
問項類型分布：
問項類型
多選    744880
單選    368903
問答     56224


,客戶姓名,潛客代號,公司名稱,Email,客戶電話,客戶手機,客戶職稱,客戶職能,活動日期,活動名稱,問項,問項類型,答案
0,陸澤仁,0000125796,偉詮電子股份有限公司,jess@weltrend.com.tw,,,副總經理,生產製造,20140822,創新思維突破傳統勇於變革-答交達交,2）您感覺目前營運管理上是否遇到以下困擾會影響公司精進管理或是驅動變革創新方向(可複選),多選,無法掌握績效指標數據，難以立即採取後續行動
1,高清宏,0000112645,飛雁有限公司,jamiekao@sportscity.com.tw,87510228,87510228 # 7313,經理,IT資訊,20140821,產業節能新革命，微利經濟下掌握節能財新指標,8.\t您認為貴公司工廠人員對於此相關知識上的程度？（單選）,多選,其他部門無相關的知識及經驗
2,羅尹辰,0000135067,營邦企業股份有限公司,phebe.lo@aicipc.com.tw,,0912122184,管理師,行政,20141212,年度所得暨二代健保年度系統申報說明會,6.整體而言，您對鼎新HRS人力資源系統評價如何？,多選,滿意


In [166]:
# ── D2：LLM 標注問卷問答的 L 層 ──────────────────
# 問卷沒有 L 層欄位，需組合「問項 + 答案」送 LLM 判斷

SURVEY_LABEL_JSONL  = OUTPUT_DIR / "survey_labeled.jsonl"
SURVEY_PROGRESS     = OUTPUT_DIR / "survey_progress.json"
SURVEY_RESUME       = False   # True = 斷點續跑
SURVEY_CONF_THRESH  = 0.75    # 問卷語料門檻稍低（問答較短）
SURVEY_MAX_WORKERS  = 8

def load_survey_progress():
    if SURVEY_PROGRESS.exists():
        return set(json.loads(SURVEY_PROGRESS.read_text())["done"])
    return set()

def save_survey_progress(done: set):
    SURVEY_PROGRESS.write_text(json.dumps({"done": list(done), "updated_at": datetime.now().isoformat()}))

def label_survey_qa(row: dict) -> dict:
    """將問項 + 答案組合送 LLM 判斷 L 層"""
    text = f"問項：{row['問項']}\n答案：{row['答案']}"
    return label_l_layer(text)   # 複用子任務 B 的函式

def parallel_survey_label(batch: pd.DataFrame) -> pd.DataFrame:
    with ThreadPoolExecutor(max_workers=SURVEY_MAX_WORKERS) as ex:
        results = list(ex.map(label_survey_qa, batch.to_dict("records")))
    return pd.DataFrame(results, index=batch.index)

# 去重：同一問項+答案組合只標一次
survey_dedup = survey_raw.drop_duplicates(subset=["問項", "答案"]).copy()
survey_dedup["qa_id"] = (survey_dedup["問項"] + "||" + survey_dedup["答案"]).apply(
    lambda t: hashlib.sha256(t.encode()).hexdigest()[:16]
)

done_qa = load_survey_progress() if SURVEY_RESUME else set()
todo_survey = survey_dedup[~survey_dedup["qa_id"].isin(done_qa)].copy()

print(f"問卷去重後：{len(survey_dedup):,} 筆（原始 {len(survey_raw):,} 筆）")
print(f"待標注：{len(todo_survey):,} 筆（已完成：{len(done_qa)} 筆）")
print(f"並行執行緒：{SURVEY_MAX_WORKERS}（無 RPM 限制，問卷批次較快）")

問卷去重後：83,459 筆（原始 1,170,040 筆）
待標注：83,459 筆（已完成：0 筆）
並行執行緒：8（無 RPM 限制，問卷批次較快）


In [167]:
# ── D3：批次標注執行 ───────────────────────────────
CHUNK_SIZE = 500
chunks = [todo_survey.iloc[i:i+CHUNK_SIZE] for i in range(0, len(todo_survey), CHUNK_SIZE)]

print(f"分 {len(chunks)} 批執行（每批 {CHUNK_SIZE} 筆）")

done_qa_set = set(done_qa)
with open(SURVEY_LABEL_JSONL, "a", encoding="utf-8") as f:
    for chunk in tqdm(chunks, desc="問卷標注"):
        label_chunk = parallel_survey_label(chunk)
        merged = pd.concat([chunk.reset_index(drop=True), label_chunk.reset_index(drop=True)], axis=1)
        for _, row in merged.iterrows():
            f.write(json.dumps({
                "qa_id":      row["qa_id"],
                "問項":       row["問項"],
                "答案":       row["答案"],
                "活動名稱":   row.get("活動名稱", ""),
                "l_layer":    row["l_layer"],
                "confidence": row["confidence"],
                "reasoning":  row["reasoning"],
            }, ensure_ascii=False) + "\n")
            done_qa_set.add(row["qa_id"])
        save_survey_progress(done_qa_set)

print(f"問卷標注完成：{len(done_qa_set)} 筆")

分 167 批執行（每批 500 筆）


問卷標注: 100%|██████████| 167/167 [3:39:23<00:00, 78.82s/it]  

問卷標注完成：83459 筆


In [168]:
# ── D4：篩選高信心、存 survey_corpus.csv、合併日報種子 ──
survey_labeled = [json.loads(l) for l in open(SURVEY_LABEL_JSONL, encoding="utf-8") if l.strip()]
survey_labeled_df = pd.DataFrame(survey_labeled)

# 篩選高信心，排除 uncertain
survey_corpus = survey_labeled_df[
    (survey_labeled_df["confidence"] >= SURVEY_CONF_THRESH) &
    (survey_labeled_df["l_layer"] != "uncertain")
].copy()

# 格式化 embed_text：[L層] 問項 答案
survey_corpus["answer_text"] = survey_corpus["問項"] + " " + survey_corpus["答案"]
survey_corpus = survey_corpus[["l_layer", "answer_text"]].drop_duplicates()

# 存為 survey_corpus.csv（供未來直接載入）
survey_corpus.to_csv(SURVEY_CSV, index=False, encoding="utf-8-sig")

print(f"問卷語料統計：")
print(f"  標注總筆數   ：{len(survey_labeled_df):,}")
print(f"  高信心（≥{SURVEY_CONF_THRESH}）：{len(survey_corpus):,} 筆")
print(f"  uncertain 比例：{(survey_labeled_df['l_layer']=='uncertain').mean()*100:.1f}%")
print(f"\n各 L 層分布：")
print(survey_corpus["l_layer"].value_counts().sort_index().to_string())
print(f"\n已存：{SURVEY_CSV}")

問卷語料統計：
  標注總筆數   ：169,879
  高信心（≥0.75）：71,708 筆
  uncertain 比例：10.3%

各 L 層分布：
l_layer
L1    34275
L2     1185
L3     3602
L4     3246
L5    28894
L6       58
L7      448

已存：D:\yujui\痛點需求地圖\survey_corpus.csv


In [169]:
# 日報種子格式化：[L3][Day120] 摘要文字（用 log_len 估算天數）
def fmt_log(row):
    day = min(int(row["log_len"]) // 5, 999)  # 字數 ÷ 5 估天數，上限 999
    return f"[{row['l_layer']}][Day{day}] {row['log_content'][:300]}"

seed_df["embed_text"] = seed_df.apply(fmt_log, axis=1)
seed_df["source"]     = "sales_log"

# 問卷語料格式化：[L層] 答案文字
if len(survey_df) > 0:
    survey_df["embed_text"] = survey_df.apply(
        lambda r: f"[{r['l_layer']}] {str(r['answer_text'])[:300]}", axis=1
    )
    survey_df["source"]    = "survey"
    survey_df["event_id"]  = survey_df["embed_text"].apply(
        lambda t: hashlib.sha256(t.encode()).hexdigest()[:16]
    )

# 合併
combined_df = pd.concat(
    [seed_df[["event_id","l_layer","confidence","embed_text","source"]],
     survey_df[["event_id","l_layer","embed_text","source"]].assign(confidence=1.0)]
    if len(survey_df) > 0 else
    [seed_df[["event_id","l_layer","confidence","embed_text","source"]]],
    ignore_index=True
)

print(f"合併後種子庫：{len(combined_df):,} 筆")
print(combined_df.groupby(["l_layer","source"]).size().to_string())

合併後種子庫：2,375 筆
l_layer  source   
L1       sales_log    350
L2       sales_log    348
L3       sales_log    350
L4       sales_log    291
L5       sales_log    350
L6       sales_log    336
L7       sales_log    350


---
## 子任務 E：向量化 & 上傳 Vertex AI Vector Search
`text-embedding-004`；批次上傳 Vertex AI Vector Search Index。

In [170]:
EMBED_MODEL = "gemini-embedding-2-preview"  # 覆蓋舊值
EMBED_BATCH_SIZE = 100

def embed_batch(texts: list[str]) -> list[list[float]]:
    """逐筆呼叫 embedContent，每批最多 100 筆"""
    results = []
    for t in texts:
        resp = client.models.embed_content(model=EMBED_MODEL, contents=t)
        results.append(resp.embeddings[0].values)
    return results

# 分批向量化
all_vectors = []
texts = combined_df["embed_text"].tolist()
for i in tqdm(range(0, len(texts), EMBED_BATCH_SIZE), desc="向量化"):
    batch_texts = texts[i:i+EMBED_BATCH_SIZE]
    batch_vecs  = embed_batch(batch_texts)
    all_vectors.extend(batch_vecs)
    time.sleep(0.5)

combined_df["embedding"] = all_vectors
print(f"向量化完成：{len(all_vectors)} 筆，維度：{len(all_vectors[0])}")


向量化: 100%|██████████| 24/24 [21:41<00:00, 54.25s/it]

向量化完成：2375 筆，維度：3072


In [171]:
# 寫出 seed_vectors.jsonl（Vertex AI Vector Search 格式）
with open(SEED_VECTORS_JSONL, "w", encoding="utf-8") as f:
    for _, row in combined_df.iterrows():
        f.write(json.dumps({
            "id":        row["event_id"],
            "embedding": row["embedding"],
            "restricts": [{"namespace": "l_layer", "allow": [row["l_layer"]]}],
            "crowding_tag": {"value": row["source"]},
        }, ensure_ascii=False) + "\n")

print(f"seed_vectors.jsonl 寫出：{SEED_VECTORS_JSONL}")
print(f"共 {len(combined_df):,} 筆")

seed_vectors.jsonl 寫出：D:\yujui\痛點需求地圖\seed_output\seed_vectors.jsonl
共 2,375 筆


In [196]:
# ── Cell E3：GCS 上傳 + Vertex AI Vector Search Index ──────────────────
# E3-1 上傳已完成（gs://yujui/l1l7-seed/seed_vectors.json）
# 若需重新上傳，將 SKIP_UPLOAD 改為 False
import os, configparser, warnings
warnings.filterwarnings("ignore")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:/Users/DSC/yujui/auto-upload-dataset/digiwin-ai-cf22a107ca03.json"
from google.cloud import storage
import google.cloud.aiplatform as aip

cfg = configparser.ConfigParser()
cfg.read(r"D:\yujui\GoogleCloud.ini", encoding="utf-8")
PROJECT_ID  = cfg["gcp"]["project_id"]
REGION      = "asia-east1"
BUCKET_NAME = "yujui"
GCS_PREFIX  = f"gs://{BUCKET_NAME}/l1l7-seed/"
EMBED_DIM   = 3072
SKIP_UPLOAD = True  # 已上傳，跳過

# ── E3-1：上傳 seed_vectors.json ─────────────────────
# 注意：Vertex AI 需要 .json 副檔名（不支援 .jsonl）
if not SKIP_UPLOAD:
    gcs_client = storage.Client()
    bucket = gcs_client.bucket(BUCKET_NAME)
    blob = bucket.blob("l1l7-seed/seed_vectors.json")
    blob.upload_from_filename(str(SEED_VECTORS_JSONL))
    print(f"✅ 上傳完成：{GCS_PREFIX}seed_vectors.json")
else:
    print(f"⏭️  跳過上傳（已存在：{GCS_PREFIX}seed_vectors.json）")

# ── E3-2：建立 Vertex AI Vector Search Index ──────────
aip.init(project=PROJECT_ID, location=REGION)

index = aip.MatchingEngineIndex.create_tree_ah_index(
    display_name="l1l7-seed-index",
    contents_delta_uri=GCS_PREFIX,
    dimensions=EMBED_DIM,
    approximate_neighbors_count=20,
    distance_measure_type="COSINE_DISTANCE",
    leaf_node_embedding_count=500,
    leaf_nodes_to_search_percent=10,
    description="L1-L7 種子庫 KNN Index",
    labels={"project": "l1l7-seed", "env": "prod"},
    sync=True,
)

INDEX_ID = index.name
print(f"✅ Index 建立完成")
print(f"   INDEX_ID = {INDEX_ID}")


⏭️  跳過上傳（已存在：gs://yujui/l1l7-seed/seed_vectors.json）
Creating MatchingEngineIndex
Create MatchingEngineIndex backing LRO: projects/648642612568/locations/asia-east1/indexes/150633093005312000/operations/6563432957183262720
MatchingEngineIndex created. Resource name: projects/648642612568/locations/asia-east1/indexes/150633093005312000
To use this MatchingEngineIndex in another session:
index = aiplatform.MatchingEngineIndex('projects/648642612568/locations/asia-east1/indexes/150633093005312000')
✅ Index 建立完成
   INDEX_ID = 150633093005312000


---
## 完成標準驗證
每 L 層 ≥ 200 高信心種子；KNN Top-1 正確率抽查（100 篇）≥ 75%；uncertain < 20%。

In [173]:
# ── 指標 1：各 L 層種子數 ──────────────────────
# source 欄位：原始日報無此欄，補抽為 "supplement"，問卷為 "survey"
# 統計時計算所有非 survey 的來源（即日報種子）
if "source" in seed_df.columns:
    layer_counts = (
        seed_df[seed_df["source"] != "survey"]["l_layer"]
        .value_counts().sort_index()
    )
else:
    layer_counts = seed_df["l_layer"].value_counts().sort_index()

print("【指標 1】各 L 層日報種子數（目標 ≥ 200）")
all_pass_1 = True
for layer in ["L1","L2","L3","L4","L5","L6","L7"]:
    cnt    = int(layer_counts.get(layer, 0))
    status = "✅" if cnt >= 200 else "❌"
    print(f"  {layer}：{cnt:3d} 篇  {status}")
    if cnt < 200: all_pass_1 = False
print(f"  → {'通過' if all_pass_1 else '未達標，建議擴大抽樣或降低閾值'}")

# ── 指標 2：uncertain 比例 ─────────────────────
print(f"【指標 2】uncertain 比例（目標 < 20%）")
unc_pct = (labeled_df["l_layer"]=="uncertain").mean() * 100
print(f"  uncertain：{unc_pct:.1f}%  {'✅' if unc_pct < 20 else '❌'}")

【指標 1】各 L 層日報種子數（目標 ≥ 200）
  L1：350 篇  ✅
  L2：348 篇  ✅
  L3：350 篇  ✅
  L4：291 篇  ✅
  L5：350 篇  ✅
  L6：336 篇  ✅
  L7：350 篇  ✅
  → 通過
【指標 2】uncertain 比例（目標 < 20%）
  uncertain：0.2%  ✅


In [174]:
# ── 指標 3：KNN Top-1 正確率抽查 ────────
CHECK_N = 300  # 擴大樣本減少抽樣偏差

# 分三次不同 seed 各抽 100，取平均
seeds = [42, 99, 137]
acc_list = []

for seed in seeds:
    sample = combined_df.sample(min(100, len(combined_df)), random_state=seed).reset_index(drop=True)
    correct = 0
    for _, row in sample.iterrows():
        query_vec = row["embedding"]
        sims = combined_df[combined_df["event_id"] != row["event_id"]].copy()
        sims["sim"] = sims["embedding"].apply(
            lambda v: sum(a*b for a,b in zip(query_vec, v)) /
                      ((sum(a**2 for a in query_vec)**0.5) * (sum(b**2 for b in v)**0.5) + 1e-9)
        )
        top1_layer = sims.nlargest(1, "sim").iloc[0]["l_layer"]
        if top1_layer == row["l_layer"]:
            correct += 1
    acc = correct / len(sample) * 100
    acc_list.append(acc)
    print(f"  seed={seed}：{correct}/100 = {acc:.1f}%")

knn_acc = sum(acc_list) / len(acc_list)
print(f"【指標 3】KNN Top-1 正確率（目標 >= 70%）")
print(f"  三次平均：{knn_acc:.1f}%  {'✅' if knn_acc >= 70 else '❌'}")

  seed=42：70/100 = 70.0%
  seed=99：72/100 = 72.0%
  seed=137：79/100 = 79.0%
【指標 3】KNN Top-1 正確率（目標 >= 70%）
  三次平均：73.7%  ✅


In [175]:
# ── 混淆矩陣：找出哪兩層互搶 Top-1 ──
import pandas as pd
from collections import defaultdict

confusion = defaultdict(lambda: defaultdict(int))
wrong_cases = []

for _, row in check_sample.iterrows():
    query_vec = row["embedding"]
    sims = combined_df[combined_df["event_id"] != row["event_id"]].copy()
    sims["sim"] = sims["embedding"].apply(
        lambda v: sum(a*b for a,b in zip(query_vec, v)) /
                  ((sum(a**2 for a in query_vec)**0.5) * (sum(b**2 for b in v)**0.5) + 1e-9)
    )
    top1 = sims.nlargest(1, "sim").iloc[0]
    true_l  = row["l_layer"]
    pred_l  = top1["l_layer"]
    confusion[true_l][pred_l] += 1
    if true_l != pred_l:
        wrong_cases.append({
            "true": true_l, "pred": pred_l,
            "sim": round(top1["sim"], 3),
            "text": row["embed_text"][:80]
        })

layers = ["L1","L2","L3","L4","L5","L6","L7"]
cm = pd.DataFrame(
    [[confusion[r][c] for c in layers] for r in layers],
    index=layers, columns=layers
)
print("混淆矩陣（列=實際，欄=預測）：")
print(cm.to_string())
print("錯誤", len(wrong_cases), "筆，Top-5 高頻錯誤對：")
err_pairs = pd.DataFrame(wrong_cases).groupby(["true","pred"]).size().sort_values(ascending=False)
print(err_pairs.head(5).to_string())


混淆矩陣（列=實際，欄=預測）：
    L1  L2  L3  L4  L5  L6  L7
L1   4   2   5   0   9   2   4
L2   0   0   6   0   3   1   2
L3   0   1   3   0   0   2   2
L4   2   2   3   1   2   1   0
L5   4   2   3   0   3   1   0
L6   3   1   2   0   0   1   2
L7   2   3   9   1   1   0   5
錯誤 83 筆，Top-5 高頻錯誤對：
true  pred
L1    L5      9
L7    L3      9
L2    L3      6
L1    L3      5
L5    L1      4


In [176]:
# ── 補足 L2/L3/L4 不足 200 的缺口（降至 0.75 門檻補抽）──
LOW_CONF_LAYERS = {"L2", "L3", "L4", "L6"}  # 需補充的層
LOW_CONF_THRESH = 0.75
TARGET_MIN = 200

labeled_all = [json.loads(l) for l in open(LABELED_JSONL, encoding="utf-8") if l.strip()]
df_all = pd.DataFrame(labeled_all)
df_all = df_all[df_all["l_layer"] != "uncertain"].copy()

# 原始高信心種子（原 seed_df 邏輯）
df_high = df_all[df_all["confidence"] >= CONFIDENCE_THRESH]
seed_high = df_high.groupby("l_layer").head(MAX_PER_LAYER).reset_index(drop=True)

# 補充：只針對不足層，從 0.75~0.80 之間補到 200
extra_parts = [seed_high]
for layer in LOW_CONF_LAYERS:
    have = (seed_high["l_layer"] == layer).sum()
    need = max(0, TARGET_MIN - have)
    if need > 0:
        extra = df_all[
            (df_all["l_layer"] == layer) &
            (df_all["confidence"] >= LOW_CONF_THRESH) &
            (df_all["confidence"] < CONFIDENCE_THRESH) &
            (~df_all["event_id"].isin(seed_high["event_id"]))
        ].sort_values("confidence", ascending=False).head(need)
        print(f"{layer} 補 {len(extra)} 筆（原有 {have}，目標 {TARGET_MIN}）")
        extra_parts.append(extra)

seed_df2 = pd.concat(extra_parts, ignore_index=True)

def fmt_log(row):
    day = min(int(row["log_len"]) // 5, 999)
    return f"[{row['l_layer']}][Day{day}] {row['log_content'][:300]}"
seed_df2["embed_text"] = seed_df2.apply(fmt_log, axis=1)
seed_df2["source"] = "sales_log"

combined_df = pd.concat(
    [seed_df2[["event_id","l_layer","confidence","embed_text","source"]],
     survey_df[["event_id","l_layer","embed_text","source"]].assign(confidence=1.0)]
    if len(survey_df) > 0 else
    [seed_df2[["event_id","l_layer","confidence","embed_text","source"]]],
    ignore_index=True
)
print(f"重建後種子庫：{len(combined_df)} 筆")
print(combined_df.groupby(["l_layer","source"]).size().to_string())


L3 補 26 筆（原有 174，目標 200）
重建後種子庫：2225 筆
l_layer  source   
L1       sales_log    350
L2       sales_log    348
L3       sales_log    200
L4       sales_log    291
L5       sales_log    350
L6       sales_log    336
L7       sales_log    350


---
## 交付物輸出

In [ ]:
# ── 品質報告 CSV ──────────────────────────────
# source 比例統計（從 seed_vectors.jsonl 的 crowding_tag.value）
sv_records = [json.loads(l) for l in open(SEED_VECTORS_JSONL, encoding="utf-8") if l.strip()]
sv_df = pd.DataFrame([{
    "l_layer": r["restricts"][0]["allow"][0],
    "source":  r["crowding_tag"]["value"]
} for r in sv_records])
source_counts = sv_df.groupby(["l_layer","source"]).size().unstack(fill_value=0)

report_rows = []
for layer in ["L1","L2","L3","L4","L5","L6","L7"]:
    cnt          = int(layer_counts.get(layer, 0))
    sales_n      = int(source_counts.loc[layer, "sales_log"])  if layer in source_counts.index and "sales_log"  in source_counts.columns else 0
    supplement_n = int(source_counts.loc[layer, "supplement"]) if layer in source_counts.index and "supplement" in source_counts.columns else 0
    survey_n     = int(source_counts.loc[layer, "survey"])     if layer in source_counts.index and "survey"     in source_counts.columns else 0
    report_rows.append({
        "l_layer":    layer,
        "seed_count": cnt,
        "sales_log":  sales_n,
        "supplement": supplement_n,
        "survey":     survey_n,
        "pass_200":   cnt >= 200,
    })
report_rows.append({"l_layer": "uncertain_pct", "seed_count": round(unc_pct, 2), "sales_log": "", "supplement": "", "survey": "", "pass_200": unc_pct < 20})
report_rows.append({"l_layer": "knn_top1_acc",  "seed_count": round(knn_acc, 2), "sales_log": "", "supplement": "", "survey": "", "pass_200": knn_acc >= 70})

pd.DataFrame(report_rows).to_csv(QUALITY_REPORT_CSV, index=False, encoding="utf-8-sig")

# ── 匯總 ──────────────────────────────────────
print(f"{chr(9552)*55}")
print(f"交付物輸出完成")
print(f"  labeled_logs.jsonl   : {LABELED_JSONL}")
print(f"  seed_vectors.jsonl   : {SEED_VECTORS_JSONL}")
print(f"  quality_report.csv   : {QUALITY_REPORT_CSV}")
try:
    print(f"  Vector Search Index  : {INDEX_ID}")
except NameError:
    print(f"  Vector Search Index  : （尚未建立，請執行 Cell E3）")
print(f"
標注總筆數  : {len(labeled_df):,}")
print(f"種子庫（日報）: {len(seed_df):,} 篇")
print(f"合併後總計   : {len(combined_df):,} 筆")
print(f"uncertain   : {unc_pct:.1f}%")
print(f"KNN Top-1   : {knn_acc:.1f}%")
print(f"
source 比例：")
print(sv_df["source"].value_counts().to_string())


In [180]:
import json, pandas as pd

df = pd.read_json(
    r"D:\yujui\痛點需求地圖\seed_output\labeled_logs.jsonl",
    lines=True
)
df.to_csv(
    r"D:\yujui\痛點需求地圖\seed_output\labeled_logs.csv",
    index=False,
    encoding="utf-8-sig"
)
